# Week 11 — Modern LLM Applications: RAG, Tools, Evaluation

Beyond the model: the systems and evaluation frameworks that make LLMs useful in practice. We build a retrieval-augmented generation pipeline from scratch, implement structured tool calling, and discuss the methodological pitfalls of modern benchmarks.

## Learning Objectives

- Build a retrieval-augmented generation (RAG) pipeline from chunking through generation, with both dense and sparse retrieval.
- Implement structured tool calling with a minimal agent loop.
- Use evaluation harnesses correctly: report confidence intervals, detect contamination, distinguish task-based from model-graded evaluation.

## Required Reading

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.
- Karpukhin, V., et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*.
- Liang, P., et al. (2022). *Holistic Evaluation of Language Models*.

In [ ]:
import sys, re, math, json, random
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
np.random.seed(0); random.seed(0)

## 1. Why retrieval-augmented generation?

A parametric LM stores knowledge in its weights — it cannot:

- cite a specific source,
- be updated without retraining,
- handle private or proprietary corpora.

**RAG** (Lewis et al., 2020) sidesteps these by retrieving relevant documents at inference time and conditioning generation on them. The architecture has four stages:

> **chunking → embedding → retrieval → generation**

We build each from scratch.

## 2. A toy knowledge base

A miniature encyclopedia, with enough redundancy and overlap that retrieval is non-trivial.

In [ ]:
KB = [
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889.",
    "Gustave Eiffel's company designed the Eiffel Tower for the 1889 World's Fair in Paris.",
    "The Statue of Liberty is a copper statue in New York Harbor, gifted by France in 1886.",
    "Frédéric Auguste Bartholdi designed the Statue of Liberty; Gustave Eiffel engineered its internal frame.",
    "The Great Wall of China stretches over 21,000 kilometers across northern China.",
    "Construction of the Great Wall began as early as the 7th century BC under various dynasties.",
    "The Colosseum is an oval amphitheatre in the centre of Rome, completed in 80 AD.",
    "The Roman Colosseum could hold an estimated 50,000 to 80,000 spectators during gladiatorial contests.",
    "Mount Everest is the Earth's highest mountain above sea level, located in the Himalayas.",
    "Mount Everest's official elevation, measured most recently in 2020, is 8,848.86 metres.",
    "Machu Picchu is a 15th-century Inca citadel located on a mountain ridge in southern Peru.",
    "Machu Picchu was built around 1450 and abandoned a century later during the Spanish conquest.",
]
print(f"KB has {len(KB)} chunks.")

## 3. Sparse retrieval — BM25

BM25 (Robertson & Walker, 1994) is the classical IR workhorse:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{f(t, d) (k_1 + 1)}{f(t, d) + k_1 (1 - b + b \cdot |d| / \bar{|d|})}.$$

With $k_1 \approx 1.5$ and $b \approx 0.75$, BM25 is a hard baseline to beat for keyword-heavy queries — and crucially, it has *no learnable parameters*.

In [ ]:
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, docs):
        self.docs_tokens = [re.findall(r"\w+", d.lower()) for d in docs]
        self.N = len(docs)
        self.avgdl = np.mean([len(d) for d in self.docs_tokens])
        # df[t] = number of docs containing t
        self.df = Counter()
        for d in self.docs_tokens:
            for t in set(d):
                self.df[t] += 1
        # Robertson-Spärck-Jones idf, lower-bounded at 0.
        self.idf = {t: max(0.0, math.log((self.N - df + 0.5) / (df + 0.5)))
                    for t, df in self.df.items()}
        # Precompute term frequencies per doc.
        self.tf = [Counter(d) for d in self.docs_tokens]
        return self

    def score(self, query, doc_idx):
        q_tokens = re.findall(r"\w+", query.lower())
        d = self.docs_tokens[doc_idx]; tf = self.tf[doc_idx]
        L = len(d)
        s = 0.0
        for t in q_tokens:
            if t not in self.idf:
                continue
            f = tf.get(t, 0)
            denom = f + self.k1 * (1 - self.b + self.b * L / self.avgdl)
            s += self.idf[t] * (f * (self.k1 + 1)) / max(denom, 1e-9)
        return s

    def search(self, query, k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: -x[1])[:k]

bm25 = BM25().fit(KB)
for q in ["Who designed the Eiffel Tower?", "How tall is Mount Everest?",
          "When was Machu Picchu built?"]:
    hits = bm25.search(q, k=2)
    print(f"\nQ: {q}")
    for i, s in hits:
        print(f"  [{s:5.2f}]  {KB[i]}")

## 4. Dense retrieval — embedding-based

Replace lexical matching with cosine similarity in an embedding space. In production, this would be a sentence-transformer model. Here, to keep the notebook self-contained, we use a deterministic hash-based bag-of-features encoder — *not* a strong retriever, but mechanically identical to the real thing.

In your project, swap the encoder for `sentence-transformers/all-MiniLM-L6-v2` and the rest of the code remains unchanged.

In [ ]:
class HashedEncoder:
    """Toy encoder: hashed bag-of-words into a fixed-dim vector. Replace with a
    real sentence encoder in practice."""
    def __init__(self, dim=128, seed=0):
        self.dim = dim
        self.rng = np.random.default_rng(seed)
        self.hashes = {}

    def _hash(self, t):
        if t not in self.hashes:
            self.hashes[t] = self.rng.normal(size=self.dim)
        return self.hashes[t]

    def encode(self, text):
        tokens = re.findall(r"\w+", text.lower())
        if not tokens:
            return np.zeros(self.dim)
        v = np.sum([self._hash(t) for t in tokens], axis=0)
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

enc = HashedEncoder(dim=256)
doc_vecs = np.stack([enc.encode(d) for d in KB])

def dense_search(query, k=3):
    q = enc.encode(query)
    sims = doc_vecs @ q
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

print("Dense retrieval:")
for q in ["Who designed the Eiffel Tower?", "How tall is Mount Everest?"]:
    print(f"\nQ: {q}")
    for i, s in dense_search(q, k=2):
        print(f"  [{s:.3f}]  {KB[i]}")

## 5. Hybrid retrieval

Dense and sparse retrieval make complementary errors. Production systems typically combine them with reciprocal rank fusion (RRF):

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}.$$

In [ ]:
def rrf(rankings, k=60, top_k=3):
    scores = Counter()
    for ranked in rankings:
        for rank, (doc_id, _) in enumerate(ranked):
            scores[doc_id] += 1.0 / (k + rank)
    return scores.most_common(top_k)

q = "Who built the iron lattice tower in Paris?"
sparse_hits = bm25.search(q, k=5)
dense_hits = dense_search(q, k=5)
fused = rrf([sparse_hits, dense_hits], top_k=3)
print(f"Q: {q}\n")
print("Sparse top-3:", [i for i, _ in sparse_hits[:3]])
print("Dense  top-3:", [i for i, _ in dense_hits[:3]])
print("Fused  top-3:", [i for i, _ in fused])
for i, _ in fused:
    print(f"  → {KB[i]}")

## 6. Putting it together: a RAG prompt template

The retrieval results are inserted into a prompt template and passed to a generator. With a real LLM API or a local model, this is one call. Here we just show the prompt construction.

In [ ]:
def rag_prompt(query, retrieved_docs):
    context = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(retrieved_docs))
    return (
        "Answer the question using only the context below. "
        "If the context does not contain the answer, say so. "
        "Cite sources by their bracketed number.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

q = "Who engineered the internal frame of the Statue of Liberty?"
hits = [i for i, _ in rrf([bm25.search(q, 5), dense_search(q, 5)], top_k=3)]
prompt = rag_prompt(q, [KB[i] for i in hits])
print(prompt)

## 7. Retrieval evaluation

The standard metrics:

- **Recall@k** — fraction of queries for which the gold passage is in the top-$k$ retrieved.
- **MRR (mean reciprocal rank)** — $\frac{1}{|Q|} \sum_q \frac{1}{\text{rank}_q}$, where $\text{rank}_q$ is the position of the first gold passage.
- **nDCG@k** — graded relevance, discounted by position.

We compute recall@k on a small annotated set.

In [ ]:
# (query, gold doc index in KB)
QUERIES = [
    ("Who designed the Eiffel Tower?", 1),
    ("When was the Eiffel Tower completed?", 0),
    ("Who engineered the Statue of Liberty's frame?", 3),
    ("How long is the Great Wall?", 4),
    ("When did Great Wall construction begin?", 5),
    ("How many spectators did the Colosseum hold?", 7),
    ("What is Mount Everest's elevation?", 9),
    ("When was Machu Picchu built?", 11),
]

def recall_at_k(retriever_fn, queries, k=3):
    hits = 0
    for q, gold in queries:
        results = retriever_fn(q, k=k)
        if gold in [r[0] for r in results]:
            hits += 1
    return hits / len(queries)

def hybrid_search(q, k=3):
    return rrf([bm25.search(q, 5), dense_search(q, 5)], top_k=k)

print(f"{'Retriever':<10} {'R@1':>6} {'R@3':>6} {'R@5':>6}")
for name, fn in [('BM25', bm25.search), ('Dense', dense_search), ('Hybrid', hybrid_search)]:
    r1 = recall_at_k(fn, QUERIES, 1)
    r3 = recall_at_k(fn, QUERIES, 3)
    r5 = recall_at_k(fn, QUERIES, 5)
    print(f"{name:<10} {r1:>6.2f} {r3:>6.2f} {r5:>6.2f}")

## 8. Structured tool calling

Modern LLMs can be prompted to emit structured JSON that names a tool and its arguments. A simple agent loop:

```
while not done:
    response = LLM(history)
    if response contains tool_call:
        result = execute_tool(tool_call)
        history += result
    else:
        return response
```

We simulate the LLM with a regex-based stub so the loop is fully runnable. The structure of the loop is identical with a real model — only the `fake_llm` function changes.

In [ ]:
TOOLS = {
    "search": lambda q: hybrid_search(q, k=2),
    "kb_get": lambda i: KB[int(i)],
    "calculator": lambda expr: eval(expr, {"__builtins__": {}}, {}),
}

def fake_llm(history):
    """Tiny scripted policy: emit search calls for factual questions,
    a calculator call if the user asks 'compute', else final answer."""
    last_user = next(m['content'] for m in reversed(history) if m['role'] == 'user')
    last_tool = next((m for m in reversed(history) if m['role'] == 'tool'), None)
    if 'compute' in last_user.lower() and last_tool is None:
        expr = re.search(r"compute\s+(.+)", last_user, re.IGNORECASE)
        return {"tool": "calculator", "args": {"expr": expr.group(1).strip().rstrip("?.")}}
    if last_tool is None:
        return {"tool": "search", "args": {"q": last_user}}
    return {"final": f"(synthesized answer using {last_tool['name']}): {str(last_tool['content'])[:200]}"}

def run_agent(user_query, max_steps=4):
    history = [{"role": "user", "content": user_query}]
    for step in range(max_steps):
        decision = fake_llm(history)
        if "final" in decision:
            print(f"  [final] {decision['final']}")
            return decision['final']
        tool, args = decision['tool'], decision['args']
        print(f"  [step {step}] call {tool}({args})")
        result = TOOLS[tool](**args)
        history.append({"role": "assistant", "content": json.dumps(decision)})
        history.append({"role": "tool", "name": tool, "content": result})
    return "(max steps reached)"

for q in ["Who designed the Eiffel Tower?", "compute 2 ** 10 - 24"]:
    print(f"\nQ: {q}")
    run_agent(q)

## 9. Evaluation pitfalls

A non-exhaustive checklist when reporting LLM benchmark numbers:

1. **Confidence intervals.** Most public benchmarks report a single number with no variance. Bootstrap the test set or use multi-seed evaluation. Liang et al. (2022) found that small models with overlapping CIs are routinely reported as "X better than Y."
2. **Contamination.** Test sets leak into pretraining corpora. Run an exact-match probe: how often does the LM complete a benchmark item that was hidden? If high, results are suspect (Magar & Schwartz, 2022).
3. **Prompt sensitivity.** Different prompt formats can change scores by tens of points. Report the prompt template used and consider an average over a few variants.
4. **Model-graded evals.** Using a stronger LLM as judge (LLM-as-judge) is fast but biases toward the judge's own preferences (Zheng et al., 2023). Cross-check against human evaluation on a sample.

We demonstrate the first two empirically.

In [ ]:
# Bootstrapped confidence interval for accuracy on a small eval set.
def bootstrap_ci(correct_flags, n_iter=2000, alpha=0.05):
    correct_flags = np.asarray(correct_flags)
    n = len(correct_flags)
    stats = []
    for _ in range(n_iter):
        idx = np.random.randint(0, n, size=n)
        stats.append(correct_flags[idx].mean())
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return correct_flags.mean(), lo, hi

# Pretend two systems both score 80% on a 20-item eval.
A = [1]*16 + [0]*4
B = [1]*16 + [0]*4
random.shuffle(A); random.shuffle(B)
acc_A, lo_A, hi_A = bootstrap_ci(A)
acc_B, lo_B, hi_B = bootstrap_ci(B)
print(f"System A: {acc_A:.2f}  95% CI [{lo_A:.2f}, {hi_A:.2f}]")
print(f"System B: {acc_B:.2f}  95% CI [{lo_B:.2f}, {hi_B:.2f}]")
print("With n=20, the CI is roughly ±0.18 — any 'A beats B by 5 points' here is noise.")

## 10. Exercises

1. **Real RAG pipeline.** Replace the `HashedEncoder` with `sentence-transformers/all-MiniLM-L6-v2`, index 1000+ documents with FAISS, and measure end-to-end answer quality (exact match) on a small QA set.
2. **Chunking ablation.** Try chunk sizes $\in \{64, 128, 256, 512\}$ tokens and overlap $\in \{0, 25, 50\}$%. Plot retrieval recall vs. chunk size. Sweet spot?
3. **Tool calling robustness.** Make the agent answer questions that need *multiple* tool calls in sequence (e.g., "compute the ratio of Everest's height to the average gladiator audience"). Track how often the agent gets it right and what failure modes appear.
4. **Contamination probe.** Pick five benchmark items from any open benchmark. Prompt a public LLM with the first half of each, see how often it completes the second half verbatim. Report the rate.

---

## Next Week

Week 12 — Capstone. End-to-end domain-specific language model project.